# Functools

> *Higher-order functions and utilities for functional programming*

## Learning Objectives

    By the end of this section, you should be able to:

    - Understand what `functools` provides for functional programming
    - Use `reduce()` to apply functions cumulatively
    - Use `partial()` and `partialmethod()` for partial application
    - Implement memoization with `lru_cache()`
    - Use `cached_property()` for caching computed properties
    - Apply `singledispatch()` and `singledispatchmethod()` for function overloading
    - Use `@wraps` properly (covered in decorators, reinforced here)
    - Recognize other useful utilities like `total_ordering`

## 1. Introduction to Functools

    The `functools` module contains tools for working with higher-order functions. It helps you:
    - Create function variants with partial application
    - Cache/memoize function results
    - Overload functions based on argument types
    - Preserve function metadata
    - Implement common functional patterns efficiently

In [20]:
# TODO: Import functools
import functools

## 2. `functools.reduce()`

    `reduce(function, iterable[, initializer])` applies a function of two arguments cumulatively to the items of an iterable, from left to right, so as to reduce the iterable to a single value.

    ```python
    reduce(f, [a, b, c, d]) = f(f(f(a, b), c), d)
    ```

In [21]:
# TODO: Basic reduce usage
from functools import reduce
import operator

# Sum
numbers = [1, 2, 3, 4, 5]
sum_result = reduce(operator.add, numbers)
print(sum_result)  # 15

# Product
prod_result = reduce(operator.mul, numbers)
print(prod_result)  # 120

# With initializer
sum_init = reduce(operator.add, numbers, 10)  # 10 + 1+2+3+4+5
print(sum_init)  # 25

15
120
25


In [22]:
# TODO: Custom reduce function
from functools import reduce

# Find max
nums = [3, 7, 2, 9, 5]
max_val = reduce(lambda x, y: x if x > y else y, nums)
print(max_val)  # 9

# Flatten list of lists
lists = [[1, 2], [3, 4], [5]]
flattened = reduce(lambda a, b: a + b, lists)
print(flattened)  # [1, 2, 3, 4, 5]

9
[1, 2, 3, 4, 5]


## 3. `functools.partial()` - Partial Application

    **Partial application** means fixing some arguments of a function and creating a new function with fewer arguments. This is a common functional programming technique.

    `partial(func, /, *args, **kwargs)` returns a new partial object which behaves like `func` when called with additional arguments.

In [23]:
# TODO: Basic partial example
from functools import partial

def multiply(x, y):
    return x * y

# Create a function that doubles
double = partial(multiply, 2)
print(double(5))  # 10
print(double(10))  # 20

# Create triple
triple = partial(multiply, 3)
print(triple(5))  # 15

10
20
15


In [24]:
# TODO: Partial with keyword arguments
from functools import partial

def power(base, exponent):
    return base ** exponent

# Square
square = partial(power, exponent=2)
print(square(4))  # 16
print(square(9))  # 81

# Cube
cube = partial(power, exponent=3)
print(cube(2))  # 8

16
81
8


In [25]:
# TODO: Partial - positional arguments fill left-to-right
from functools import partial

def greet(greeting, name, punctuation='!'):
    return f"{greeting}, {name}{punctuation}"

say_hi = partial(greet, "Hi")
print(say_hi("Alice"))  # Hi, Alice!
print(say_hi("Bob", punctuation="."))  # Hi, Bob.

say_hello_to = partial(greet, "Hello", punctuation="!!!")
print(say_hello_to("Charlie"))

Hi, Alice!
Hi, Bob.
Hello, Charlie!!!


### 3.1 `functools.partialmethod()`

    Similar to `partial()`, but designed for use as a method definition in classes.

In [26]:
# TODO: partialmethod example
from functools import partialmethod

class Greeter:
    def greet(self, greeting, name):
        return f"{greeting}, {name}!"

    say_hello = partialmethod(greet, "Hello")
    say_hi = partialmethod(greet, "Hi")

g = Greeter()
print(g.say_hello("Alice"))  # Hello, Alice!
print(g.say_hi("Bob"))  # Hi, Bob!

Hello, Alice!
Hi, Bob!


## 4. Caching with `@lru_cache`

    `@lru_cache(maxsize=128, typed=False)` is a decorator that memoizes function results. It stores the results of expensive function calls and returns cached results for the same arguments.

    **LRU** = Least Recently Used (evicts least recently used items when cache is full).

    This is extremely useful for recursive functions and expensive computations.

In [27]:
# TODO: Fibonacci without cache (slow for large n)
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

# Try small n
print([fibonacci(i) for i in range(10)])

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


In [28]:
# TODO: Fibonacci with lru_cache (fast!)
from functools import lru_cache

@lru_cache(maxsize=None)  # None means unlimited cache
def fibonacci_fast(n):
    if n <= 1:
        return n
    return fibonacci_fast(n-1) + fibonacci_fast(n-2)

print([fibonacci_fast(i) for i in range(10)])
# Check cache info
print(fibonacci_fast.cache_info())

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
CacheInfo(hits=16, misses=10, maxsize=None, currsize=10)


### Cache Management

    `lru_cache` provides methods:
    - `cache_info()` - returns stats (hits, misses, maxsize, currsize)
    - `cache_clear()` - clears the cache

In [29]:
# TODO: Cache management
from functools import lru_cache

@lru_cache(maxsize=32)
def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n-1)

factorial(10)
factorial(5)
print(factorial.cache_info())
factorial.cache_clear()
print(factorial.cache_info())

CacheInfo(hits=1, misses=10, maxsize=32, currsize=10)
CacheInfo(hits=0, misses=0, maxsize=32, currsize=0)


## 5. `@functools.cached_property`

    `@cached_property` is a decorator that transforms a method into a property whose value is computed once and then cached as a regular attribute. It's similar to `@property` but with caching.

    **Note**: Available in Python 3.8+. The cached value is stored in the instance's `__dict__`.

In [30]:
# TODO: cached_property example
from functools import cached_property
import time

class DataProcessor:
    def __init__(self, data):
        self.data = data

    @cached_property
    def expensive_result(self):
        print("Computing... (this happens once)")
        time.sleep(0.1)  # Simulate expensive computation
        return sum(self.data) ** 2

dp = DataProcessor([1, 2, 3, 4, 5])
print(dp.expensive_result)  # Computes and caches
print(dp.expensive_result)  # Returns cached value (no computation)
print(dp.expensive_result)  # Still cached

Computing... (this happens once)
225
225
225


## 6. `@singledispatch` - Single Dispatch Generic Functions

    Single dispatch allows you to write generic functions that behave differently based on the type of the **first argument**. This is Python's way of supporting function overloading.

    `@singledispatch` decorates a base function, and you register specialized implementations with `.register(type)`. 

In [31]:
# TODO: singledispatch example
from functools import singledispatch

@singledispatch
def fun(arg, verbose=False):
    """Default implementation"""
    if verbose:
        print(f"Let me just say, {arg}")
    return arg

@fun.register
def _(arg: int, verbose=False):
    if verbose:
        print(f"Strength in numbers, {arg}")
    return arg ** 2

@fun.register
def _(arg: list, verbose=False):
    if verbose:
        print(f"Enumerate this:")
    return len(arg)

@fun.register(str)
def _(arg: str, verbose=False):
    if verbose:
        print(f"String! {arg.upper()}")
    return arg.upper()

print(fun("hello"))  # HELLO
print(fun(5))  # 25
print(fun([1, 2, 3]))  # 3
print(fun(3.14))  # 3.14 (default)

HELLO
25
3
3.14


### 6.1 `@singledispatchmethod`

    Same as `singledispatch`, but for methods in classes. Dispatch is based on the type of the **first non-self/cls argument**.

In [32]:
# TODO: singledispatchmethod example
from functools import singledispatchmethod

class Formatter:
    @singledispatchmethod
    def format(self, arg):
        return str(arg)

    @format.register
    def _(self, arg: int):
        return f"Integer: {arg}"

    @format.register
    def _(self, arg: float):
        return f"Float: {arg:.2f}"

    @format.register(str)
    def _(self, arg: str):
        return f"String: '{arg}'"

f = Formatter()
print(f.format(10))
print(f.format(3.14159))
print(f.format("hello"))
print(f.format([1,2,3]))

Integer: 10
Float: 3.14
String: 'hello'
[1, 2, 3]


## 7. `@functools.wraps` (Quick Recap)

    As we saw in decorators, `wraps` preserves the metadata of the original function (name, docstring, etc.).

In [33]:
# TODO: wraps example
from functools import wraps

def my_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        """Wrapper docstring"""
        print("Calling...")
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def hello():
    """Say hello"""
    print("Hello!")

hello()
print(hello.__name__)  # 'hello' (not 'wrapper')
print(hello.__doc__)   # 'Say hello' (not 'Wrapper docstring')

Calling...
Hello!
hello
Say hello


## 8. Other Useful Functools Utilities (Brief Overview)

    - `@total_ordering` - Generates missing comparison methods (`__le__`, `__ge__`, `__lt__`, `__gt__`) if you define `__eq__` and one other
    - `cmp_to_key` - Converts an old-style cmp function to a key function for `sorted()`
    - `update_wrapper` - Like `wraps` but called directly
    - `RLock` - Recursive lock (in other contexts)

    These are less commonly used in basic FP but good to know.

## 9. Exercises

    Practice using functools utilities.

### Exercise 1: Using `reduce`

    Use `functools.reduce()` to find the greatest common divisor (GCD) of a list of numbers, or alternatively to compute the factorial of n.

In [34]:
# TODO: Use reduce to compute factorial


### Exercise 2: Partial Application

    Create a partial function `round_to_2` that rounds numbers to 2 decimal places using `round()`.

In [35]:
# TODO: Create round_to_2 with partial


### Exercise 3: LRU Cache for Expensive Function

    Write a function that computes the sum of squares up to n: `sum_squares(n) = 1² + 2² + ... + n²`. Add `@lru_cache` and test it.

In [36]:
# TODO: sum_squares with lru_cache


### Exercise 4: Custom Cache with cached_property

    Create a class `Circle` with radius. Add a `@cached_property` for `area` (π*r²) and `perimeter` (2*π*r).

In [37]:
# TODO: Circle class with cached_property
# import math


### Exercise 5: Single Dispatch

    Implement a generic `process()` function using `@singledispatch` that:
    - For `int`: returns the number * 2
    - For `str`: returns the string reversed
    - For `list`: returns the sum of elements
    - For default: returns the type of the argument as string

In [38]:
# TODO: Implement singledispatch process function


## 10. Summary

    - `functools` provides essential tools for functional programming in Python.
    - `reduce()` collapses an iterable to a single value by applying a binary function cumulatively.
    - `partial()` enables **partial application** - creating specialized functions from general ones.
    - `lru_cache()` adds **memoization** for performance optimization, especially in recursion.
    - `cached_property()` caches computed property values on instances.
    - `singledispatch()` allows **type-based function overloading** (generic functions).
    - `wraps()` preserves function metadata when decorating - critical for debugging/introspection.
    - These tools promote **reusability, performance, and cleaner functional code**.